# Tai lap PHUONG PHAP CO SO Forget-MI (Kaggle, 30 epoch)

Chay `training/forgetmi_partial.py` - **ban tai lap TRUNG THANH code goc Forget-MI**
(giu nguyen moi hanh vi goc, chi sua cac loi khien code khong chay duoc).

**Vi sao chay lai:** so cu (Df-AUC 0.561) do tren **split CU (toan bo test)**, khong so duoc voi
P3-P6/main (do tren `D_t_final` = 75% test). Cell 4 eval lai model tren **DUNG split moi**
=> ra 1 dong **so sanh duoc** trong `results_advanced.csv`.

**Uoc tinh:** ~3h (30 epoch, full model 113M params, co monitor CosSim moi epoch nhu ban goc).

---
### Cac loi GOC da sua (chi de code chay duoc, KHONG them logic)
| Loi trong code goc | Sua |
|---|---|
| `AlignedSampler`: `torch.randperm(...)` khong gan bien -> shuffle vo hieu | gan lai ket qua |
| Import `from joint_embedding import ...` (sai path) | `from training.joint_embedding import ...` |
| `evaluate()` goi ham chua dinh nghia | giu nguyen trang thai comment nhu goc |
| Thieu CLI/config path cho Kaggle | them `--config/--seed/--override` (khong doi logic train) |

### Hanh vi GOC duoc GIU NGUYEN (khong sua - de trung thanh)
- Gate tao MOI moi batch (random, khong nam trong optimizer) -> `L_md`/`L_mkr` la nhieu ngau nhien
- `og_frgt_joint_emb` dung nham `gate_ul_frgt` + truyen img 2 lan -> nhung **khong duoc dung** o loss nao
- Epoch 0 KHONG backward (chi calibrate margin); `optimizer.step()` **1 lan/epoch**
- `model_og.train()` (BatchNorm dung batch-stats)
- `use_noise=false` -> `L_uu = -distance` (day-ra-xa KHONG chan)


In [ ]:
# Cell 1: setup repo + deps
import os, subprocess
WORK_DIR = '/kaggle/working'
REPO_DIR = f'{WORK_DIR}/Forget-MI-LoKU'
REPO_URL = 'https://github.com/nhnhu146/Forget-MI-LoKU.git'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
assert os.path.exists('training/forgetmi_partial.py'), 'Khong thay training/forgetmi_partial.py'
subprocess.run(['pip', 'install', '-q', 'pydicom', 'scikit-image', 'scikit-learn',
                'pyyaml', 'wandb', 'seaborn==0.13.2'], check=True)
subprocess.run(['pip', 'install', '-q', 'transformers==4.38.0', 'peft==0.10.0',
                'accelerate==0.27.0'], check=True)
import torch
assert torch.cuda.is_available(), 'Bat GPU trong Kaggle Settings truoc khi chay.'
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPU   :', torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CO + path discovery
import glob, os

FORGET_PCT = 3        # 3 | 6 | 10
SEED       = 42

# Eval lai tren split moi (D_t_final) de so duoc voi P3-P6/main
EVAL_ON_NEW_SPLIT = True
EVAL_VAL_BEST     = True    # eval them checkpoint val-best (ngoai 'last')

assert FORGET_PCT in (3, 6, 10)

def find_dataset(*slugs):
    for slug in slugs:
        direct = f'/kaggle/input/{slug}'
        if os.path.isdir(direct):
            return direct
        hits = glob.glob(f'/kaggle/input/datasets/*/{slug}')
        if hits:
            return sorted(hits)[0]
    return None

def first_existing(root, relatives):
    for rel in relatives:
        p = os.path.join(root, rel)
        if os.path.exists(p):
            return p
    return None

DATA_ROOT = find_dataset('forget-mi-data')
MODELS_ROOT = find_dataset('forget-mi-models-full', 'forget-mi-models-v2', 'forget-mi-models')
assert DATA_ROOT and MODELS_ROOT, 'Add forget-mi-data va forget-mi-models(-full) vao Kaggle.'

base_hits = glob.glob(os.path.join(MODELS_ROOT, '**', 'training_original_model', 'pytorch_model.bin'), recursive=True)
gold_hits = glob.glob(os.path.join(MODELS_ROOT, '**', f'model_retrained_{FORGET_PCT}per', '**', 'pytorch_model.bin'), recursive=True)
BASE_MODEL = os.path.dirname(sorted(base_hits, key=len)[0]) if base_hits else None
GOLD_MODEL = os.path.dirname(sorted(gold_hits, key=len)[0]) if gold_hits else BASE_MODEL
TEXT_DIR = first_existing(DATA_ROOT, ['data/metadata', 'metadata'])
IMG_DIR  = first_existing(DATA_ROOT, ['data/img_data', 'img_data'])
FORGET_CSV = f'./data_splits/forget_set_{FORGET_PCT}per.csv'

RUN_ID       = f'forgetmi_baseline_{FORGET_PCT}per_s{SEED}'
OUTPUT_DIR   = f'/kaggle/working/baseline_output/{FORGET_PCT}per_s{SEED}'
BASE_CSV     = '/kaggle/working/results_baseline_native.csv'   # eval GOC cua baseline (split cu)
RESULTS_CSV  = '/kaggle/working/results_advanced.csv'          # eval MOI (D_t_final) - SO DUOC

for name, path in {'base model': BASE_MODEL, 'text metadata': TEXT_DIR,
                   'images': IMG_DIR, 'forget csv': FORGET_CSV}.items():
    assert path and os.path.exists(path), f'Missing {name}: {path}'

COMMON_OVR = {
    'forget_set_path': FORGET_CSV,
    'base_model_path': BASE_MODEL,
    'bert_pretrained_dir': BASE_MODEL,
    'retrained_model_path': GOLD_MODEL,
    'text_data_dir': TEXT_DIR,
    'img_data_dir': IMG_DIR,
}
print('FORGET_PCT :', FORGET_PCT, '| SEED', SEED)
print('BASE_MODEL :', BASE_MODEL)
print('GOLD_MODEL :', GOLD_MODEL if gold_hits else 'N/A')
print('OUTPUT_DIR :', OUTPUT_DIR)


In [ ]:
# Cell 3: CHAY TAI LAP FORGET-MI (30 epoch, ~3h)
# evaluate_last_and_best=1 -> chi luu checkpoints/last.pt + val_best.pt (~900MB)
# thay vi 30 x 450MB = 13.5GB (ban goc luu MOI epoch -> tran dia Kaggle).
# KHONG doi logic train, chi doi cach LUU checkpoint.
import os, subprocess, time

ovr = dict(COMMON_OVR)
ovr.update({
    'output_dir': OUTPUT_DIR,
    'results_csv_path': BASE_CSV,
    'evaluate_last_and_best': 1,
    'id': RUN_ID,
})
arg = ','.join(f'{k}={v}' for k, v in ovr.items())
env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_MODE': 'disabled'}
cmd = ['python', 'training/forgetmi_partial.py', '--config', 'config_baseline_kaggle.yaml',
       '--seed', str(SEED), '--fresh', '--override', arg]
print('=' * 70); print('RUN', RUN_ID); print('=' * 70)
t0 = time.time()
try:
    subprocess.run(cmd, env=env, check=True)
    print(f'DONE baseline ({(time.time()-t0)/3600:.2f}h)')
except subprocess.CalledProcessError as e:
    print(f'FAILED baseline rc={e.returncode}')


In [ ]:
# Cell 4: EVAL LAI tren D_t_final (=> so duoc voi P3-P6/main)
# Dung CUNG pipeline eval/split voi cac phuong phap moi.
import os, glob, subprocess

def eval_ckpt(ckpt, label, kind):
    if not os.path.exists(ckpt):
        print('BO QUA (khong thay):', ckpt); return
    ovr = dict(COMMON_OVR); ovr['results_csv_path'] = RESULTS_CSV
    arg = ','.join(f'{k}={v}' for k, v in ovr.items())
    env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_MODE': 'disabled'}
    cmd = ['python', 'training/forgetmi_eval_only.py', '--config', 'config_advanced_kaggle.yaml',
           '--seed', str(SEED), '--label', label, '--model_type', 'state_dict',
           '--model_path', ckpt, '--method', 'forgetmi', '--override', arg]
    print(f'{chr(10)}--- eval {label} ({kind}) ---')
    try:
        subprocess.run(cmd, env=env, check=True)
    except subprocess.CalledProcessError as e:
        print(f'FAILED eval {label} rc={e.returncode}')

if EVAL_ON_NEW_SPLIT:
    cp = os.path.join(OUTPUT_DIR, 'checkpoints')
    eval_ckpt(os.path.join(cp, 'last.pt'), f'forgetmi_last_{FORGET_PCT}per', 'last E30')
    if EVAL_VAL_BEST:
        eval_ckpt(os.path.join(cp, 'val_best.pt'), f'forgetmi_valbest_{FORGET_PCT}per', 'val-best')
else:
    print('EVAL_ON_NEW_SPLIT=False -> bo qua.')


In [ ]:
# Cell 5: xem ket qua
import os, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

print('===== EVAL GOC cua baseline (split CU - chi de doi chieu tai lap) =====')
if os.path.exists(BASE_CSV):
    df0 = pd.read_csv(BASE_CSV)
    cols0 = [c for c in ['id','Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','MIA_paper',
                         'forget_ce','test_ce','unlearn_core_hours'] if c in df0.columns]
    print(df0[cols0].to_string(index=False))
else:
    print('chua co', BASE_CSV)

print(chr(10) + '===== EVAL MOI tren D_t_final (SO DUOC voi P3-P6/main) =====')
if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    cols = [c for c in ['method','checkpoint_kind','id','Forget_AUC','Forget_Macro_F1',
                        'Test_AUC','Test_Macro_F1','MIA','MIA_paper','forget_ce','test_ce'] if c in df.columns]
    print(df[cols].to_string(index=False))
else:
    print('chua co', RESULTS_CSV)

print(chr(10) + 'TAI VE: results_advanced.csv + results_baseline_native.csv tu tab Output.')
